In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import numpy as np

from src.robustness.milp_functions import *
from src.architectures.networkArchitectures import networkRegistry
from src.testing.networkTesting import testNetwork
from src.training.networkTraining import trainNetwork, saveNetwork
from src.utils.loadNetwork import loadNetwork

In [2]:
# Train networks in registry
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

numEpochs = 2

for name, entry in networkRegistry.items():
    print(f"\nTraining {name}...")

    model = entry.NetworkClass()

    durations = trainNetwork(model, numEpochs, device)
    saveNetwork(model, name)

    print(f"Finished {name} | epoch times: {durations}")



Training Dense1x10...
Finished Dense1x10 | epoch times: [8.895484999986365, 8.153822599968407]

Training Dense1x15...
Finished Dense1x15 | epoch times: [8.34115799999563, 8.216839200002141]

Training Dense1x20...
Finished Dense1x20 | epoch times: [8.284108499996364, 8.258437999989837]

Training Dense1x25...
Finished Dense1x25 | epoch times: [8.20383320003748, 8.84597029996803]

Training Dense1x30...
Finished Dense1x30 | epoch times: [8.750416899973061, 7.958555100020021]

Training Dense1x35...
Finished Dense1x35 | epoch times: [8.09238709998317, 8.303544499969576]

Training Dense1x40...
Finished Dense1x40 | epoch times: [8.067583099997137, 7.914767100010067]

Training Dense1x45...
Finished Dense1x45 | epoch times: [8.149529499991331, 7.954611699969973]

Training Dense1x50...
Finished Dense1x50 | epoch times: [8.064723599993158, 7.856073899951298]

Training Dense1x55...
Finished Dense1x55 | epoch times: [8.035130200034473, 7.91946939995978]

Training Dense1x60...
Finished Dense1x60 | e

In [3]:
DETAILED_TEST = True

for name in networkRegistry:
    print(f"----- Testing {name} -----")
    network = loadNetwork(name)
    correctClassifications, totalClassifications = testNetwork(network = network)

    if DETAILED_TEST:
        for number, correctCount in correctClassifications.items():
            accuracy = 100 * float(correctCount) / totalClassifications[number]
            print(f"Accuracy for number: {number} is {accuracy:.1f} %")

    sumOfCorrectClassifications = sum(correctClassifications.values())
    sumOfTotalClassifications = sum(totalClassifications.values())
    accuracy = 100 * float(sumOfCorrectClassifications) / sumOfTotalClassifications
    print(f"Total accuracy is {accuracy:.1f} %")

----- Testing Dense1x10 -----
Accuracy for number: 0 is 98.1 %
Accuracy for number: 1 is 97.7 %
Accuracy for number: 2 is 91.2 %
Accuracy for number: 3 is 89.8 %
Accuracy for number: 4 is 91.0 %
Accuracy for number: 5 is 81.2 %
Accuracy for number: 6 is 94.9 %
Accuracy for number: 7 is 91.2 %
Accuracy for number: 8 is 88.1 %
Accuracy for number: 9 is 92.3 %
Total accuracy is 91.7 %
----- Testing Dense1x15 -----
Accuracy for number: 0 is 97.8 %
Accuracy for number: 1 is 96.2 %
Accuracy for number: 2 is 90.3 %
Accuracy for number: 3 is 90.6 %
Accuracy for number: 4 is 95.1 %
Accuracy for number: 5 is 85.3 %
Accuracy for number: 6 is 94.5 %
Accuracy for number: 7 is 88.9 %
Accuracy for number: 8 is 92.2 %
Accuracy for number: 9 is 90.7 %
Total accuracy is 92.2 %
----- Testing Dense1x20 -----
Accuracy for number: 0 is 98.7 %
Accuracy for number: 1 is 97.6 %
Accuracy for number: 2 is 93.0 %
Accuracy for number: 3 is 90.0 %
Accuracy for number: 4 is 92.4 %
Accuracy for number: 5 is 87.6 %
Ac

In [4]:
# Configuration
ACCURACY_THRESHOLD = 90.0
CLASSES_TO_CHECK = [0,1,3] 

performance_results = []

for name in networkRegistry:
    network = loadNetwork(name)
    correctClassifications, totalClassifications = testNetwork(network=network)
    target_classes = CLASSES_TO_CHECK if CLASSES_TO_CHECK is not None else correctClassifications.keys()    
    passed_all_classes = True
    for number in target_classes:
        accuracy = 100 * float(correctClassifications[number]) / totalClassifications[number]
        if accuracy < ACCURACY_THRESHOLD:
            passed_all_classes = False
            break
            
    performance_results.append(passed_all_classes)

# Convert to numpy array
boolean_array = np.array(performance_results)

# Print in a copy-pasteable format
print(f"np.array({repr(boolean_array.tolist())})")

np.array([False, True, True, True, True, True, True, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, False, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True, True, False, False, False, True, True, True, True, True, True, True, True, True, True])
